# Vision Mamba (Vim), 2024

## Motivation

Vision Mamba (Vim) extends Mamba to computer vision by replacing
self-attention with **bidirectional Selective State Space Models
(SSMs)**. The goal is to preserve the representation power of Vision
Transformers while achieving **linear sequence complexity** and lower
memory usage.

------------------------------------------------------------------------

## Background

### State Space Models (SSMs)

-   Efficient for long-sequence modeling.
-   Linear complexity with sequence length.
-   Mamba introduces input-dependent selection through **B(x)**,
    **C(x)**, and **Δ(x)**.

### Limitations of Mamba for Vision

-   Unidirectional sequence modeling.
-   No explicit spatial awareness.
-   Images require positional information and global context.

------------------------------------------------------------------------

## Main Idea

-   Bidirectional Selective SSMs.
-   Position embeddings.
-   Pure sequence-based visual backbone without self-attention.

------------------------------------------------------------------------

## Vim Architecture

<div>
    <img src="../images/VISIONMAMBA.png" width="1000">
</div>

``` text
Image
  │
Patch Embedding
  │
Position Embedding + Class Token
  │
L × Vim Blocks
  │
LayerNorm
  │
MLP Head
  │
Prediction
```

------------------------------------------------------------------------

## Vim Block

1.  **LayerNorm:** Normalize the input token sequence.
2.  **Linear Projection:** Split features into **x** (SSM branch) and
    **z** (gating branch), expanding **D → E**.
3.  **Forward & Backward Processing:** Process **x** independently in
    both directions.
4.  **1-D Convolution:** Extract local spatial features.
5.  **Parameter Projection:** Generate **B(x)**, **C(x)** and **Δ(x)**
    (written as **Bₒ**, **Cₒ** and **Δₒ** in the paper, where *o*
    denotes the processing direction).
6.  **Discretization:** Use **Δₒ** to discretize the continuous SSM,
    transforming **A → Āₒ** and **Bₒ → B̄ₒ**.
7.  **Selective SSM:** Compute the forward and backward hidden states
    using **Āₒ**, **B̄ₒ** and **Cₒ**.
8.  **Gating:** Modulate both outputs using **z**.
9.  **Fusion:** Sum the forward and backward outputs.
10. **Output Projection:** Project **E → D** and apply the residual
    connection.

------------------------------------------------------------------------

## Bidirectional SSM

Unlike Mamba, Vim processes the token sequence in both directions and
fuses the two outputs, enabling richer global context modeling.

------------------------------------------------------------------------

## Position Embedding

Position embeddings provide explicit spatial information, improving
visual understanding and dense prediction.

------------------------------------------------------------------------

## Efficiency

-   Linear complexity with sequence length.
-   IO-efficient implementation inherited from Mamba.
-   Recomputation strategy reduces GPU memory.
-   Scales efficiently to high-resolution images.

------------------------------------------------------------------------

## ImageNet Results

<div>
    <img src="../images/VMAMBA_T1.png" width="400">
    <img src="../images/VMAMBA_FIG1.png" width="900">
</div>

-   Outperforms CNN-, Transformer-, and previous SSM-based backbones.
-   Higher accuracy than DeiT with comparable model sizes.
-   Similar accuracy to S4ND-ViT using about **3× fewer parameters**.
-   At **1248×1248**, Vim is **2.8× faster** than DeiT while reducing
    GPU memory by **86.8%**.

------------------------------------------------------------------------

## Semantic Segmentation

-   Consistently outperforms DeiT on ADE20K.
-   Achieves similar performance to ResNet-101 with nearly **2× fewer
    parameters**.

------------------------------------------------------------------------

## Object Detection & Instance Segmentation

-   Outperforms DeiT on COCO for both tasks.
-   Better performance on medium and large objects.
-   Enables high-resolution sequence modeling without window attention
    or other 2D priors.

------------------------------------------------------------------------

## Conclusion

Vision Mamba shows that self-attention is not necessary for strong
visual representation learning. By combining **bidirectional Selective
SSMs**, **position embeddings**, and **linear complexity**, Vim achieves
competitive accuracy with substantially better efficiency.